In [1]:
import os
from dotenv import load_dotenv

load_dotenv(
    r"C:\Users\guruz\OneDrive\Desktop\Lanchan\.env",
    override=True
)

api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError("API key not found!")

print("API key loaded successfully")

API key loaded successfully


In [2]:
from langchain_openai import ChatOpenAI

llm_openai = ChatOpenAI(
    model="gpt-5-mini",
    temperature=0
)

In [3]:
## Load Doument
from langchain_community.document_loaders import PyPDFLoader

pdf_files = [
    "./doc/01_Traffic_Rules_and_Road_Safety.pdf",
    "./doc/02_Driving_Licence_Process.pdf",
    "./doc/03_School_Policy.pdf",
    "./doc/04_Driving_School_SOP.pdf",
    "./doc/05_FAQ.pdf"
]

docs = []

for pdf_file in pdf_files:
    loader = PyPDFLoader(pdf_file)
    documents = loader.load()
    for doc in documents:
        doc.metadata={
            "document_name": pdf_file.split("/")[-1].replace(".pdf", ""),
            "source": pdf_file,
            "page": doc.metadata["page"],
            "category": "driving_school",
            "document_type": "knowledge_base"
        }
    docs.extend(documents)

    print(f"Loaded: {pdf_file} | Pages: {len(documents)}")

print("\nTotal pages loaded:", len(docs))

C:\Users\guruz\AppData\Local\Temp\ipykernel_22156\2690678501.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Loaded: ./doc/01_Traffic_Rules_and_Road_Safety.pdf | Pages: 5
Loaded: ./doc/02_Driving_Licence_Process.pdf | Pages: 5
Loaded: ./doc/03_School_Policy.pdf | Pages: 6
Loaded: ./doc/04_Driving_School_SOP.pdf | Pages: 7
Loaded: ./doc/05_FAQ.pdf | Pages: 4

Total pages loaded: 27


Convert in to chunks

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

spliter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)
chunk = spliter.split_documents(docs)


In [5]:
print(type(docs))
print(type(docs[0]))
print(docs[0].metadata)

<class 'list'>
<class 'langchain_core.documents.base.Document'>
{'document_name': '01_Traffic_Rules_and_Road_Safety', 'source': './doc/01_Traffic_Rules_and_Road_Safety.pdf', 'page': 0, 'category': 'driving_school', 'document_type': 'knowledge_base'}


## Create a Embedding Model


In [6]:
from langchain_openai import OpenAIEmbeddings

embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

In [7]:
from langchain_community.vectorstores import Chroma
vector_db = Chroma.from_documents(
    embedding=embedding_model,
    documents=chunk,
    persist_directory="./vector/"
)


In [8]:
context = vector_db.similarity_search("til me about driving school")

In [9]:
llm_openai.invoke(f"till me about driving school using this context: {context}").content

'Here’s a concise summary of the driving school information from the documents you provided.\n\nOverview\n- The school has been operating since about 2012–13.\n- It is required to hold a licence from the RTO (licence issued in Form 11). The original licence should be displayed in the school office — you may ask to see it.\n\nAdmission & eligibility\n- Minimum ages:\n  - 16 years for a motorcycle up to 50 cc (guardian consent required).\n  - 18 years for a motorcycle with gear or a car.\n- Students under 18 must have a parent or guardian sign the enrolment form (and give consent where required by law).\n\nDocuments required for admission\n- Proof of age.\n- Proof of address.\n- Recent passport-size photographs.\n- Working mobile number and an email address.\n- All documents must show the same name and date of birth; staff will check originals.\n\nCourse completion & certificate\n- On completing the course, students receive a driving school certificate (Form 5, rule 27(d)).\n- That Form 

In [21]:
from langchain_community.tools import tool

def search_driving_school_knowledge(query:str)->str:
     """
     Search the driving school's PDF knowledge base.

     Use this tool to answer questions about:
     traffic rules, road safety, driving licence process,
     school policies, SOP and FAQs.

     Always use this tool before answering the user.
     """

     results = vector_db.similarity_search(query,k=4)

     if not results:
        return "NO_RELEVANT_INFORMATION_FOUND"

     context =[]
     for i,doc in enumerate(results,start=1):
          source = doc.metadata.get("source", "Unknown")
          page = doc.metadata.get("page", "Unknown")
          category = doc.metadata.get("category", "Unknown")
          context.append(
               f"""

               Content:
               {doc.page_content}
               """
          )
     return "\n\n".join(context)

In [22]:
from langchain_core.prompts import ChatPromptTemplate

my_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
        You are a Driving School AI Assistant.

        Your responsibility is to answer questions about the driving school.

        STRICT RULES:

        1. Always use the search_driving_school_knowledge tool
           before answering.

        2. Answer ONLY using information returned by the tool.

        3. Never use your internal knowledge or general knowledge.

        4. If the retrieved information does not contain the answer,
           respond:
           "Sorry, I could not find this information in the
           driving school's knowledge base."

        5. Do not make assumptions or invent information.

        6. Keep answers simple and beginner-friendly.

        7. Mention the document source and page number whenever possible.

        8. If a question is unrelated to driving school,
           politely explain that you can only answer
           driving-school-related questions.
        """
    ),
    (
        "human",
        "{question}"
    )
])

In [17]:
chain = my_prompt | llm_openai

response = chain.invoke({
    "question": "What info can you told...?"
})

print(response.content)

Calling the driving school knowledge base to find information about "question"...


In [23]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm_openai,
    tools=[search_driving_school_knowledge],
    system_prompt="""
    You are a Driving School AI Assistant.

    Always call search_driving_school_knowledge
    before answering.

    Answer only using the retrieved information.

    Never invent answers.

    If information is unavailable, say:
    Sorry, I could not find this information
    in the driving school's knowledge base.

    Mention source and page number whenever possible.
    """
)

response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What is the driving licence process?"
        }
    ]
})

print(response["messages"][-1].content)

Sorry, I could not find this information
in the driving school's knowledge base.
